# SurfaceTarget-CRC — phase 01 — patch 1 6

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SurfaceTarget-CRC
# Phase 1 — Patch 1.6
#
# Preliminary MSI Cohort Classification
#
# Scientific objective:
# Identify patients with concordant low MSI scores
# using published MANTIS and MSIsensor thresholds.
#
# IMPORTANT:
# Concordant low MSI is NOT equivalent to independently
# verified MSS status.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import json

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/SurfaceTarget-CRC")

METADATA_DIR = PROJECT_ROOT / "data" / "metadata"

RESULTS_DIR = PROJECT_ROOT / "results" / "tables"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MSI_PATH = (
    METADATA_DIR /
    "tcga_pancancer_msi_clinical_corrected.csv"
)

INVENTORY_PATH = (
    METADATA_DIR /
    "tcga_rnaseq_file_inventory.csv"
)

# Published default thresholds

MANTIS_THRESHOLD = 0.4
MSISENSOR_THRESHOLD = 3.5

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.6 — PRELIMINARY MSI COHORT CLASSIFICATION")
print("=" * 65)

# ------------------------------------------------------------
# 2. Load data
# ------------------------------------------------------------

msi = pd.read_csv(MSI_PATH)

inventory = pd.read_csv(
    INVENTORY_PATH,
    dtype=str
)

print("\nMSI records loaded:", len(msi))

print(
    "GDC RNA-seq patients:",
    inventory["patient_id"].nunique()
)

# ------------------------------------------------------------
# 3. Validate MSI data
# ------------------------------------------------------------

required_columns = [
    "patientId",
    "sampleId",
    "clinicalAttributeId",
    "numeric_value"
]

missing = [
    col for col in required_columns
    if col not in msi.columns
]

if missing:
    raise RuntimeError(
        f"Missing required MSI columns: {missing}"
    )

msi["numeric_value"] = pd.to_numeric(
    msi["numeric_value"],
    errors="raise"
)

attributes = {
    "MSI_SCORE_MANTIS",
    "MSI_SENSOR_SCORE"
}

if set(msi["clinicalAttributeId"]) != attributes:
    raise RuntimeError(
        "Unexpected MSI attributes."
    )

# Ensure one measurement per patient per method.

duplicates = msi.duplicated(
    subset=["patientId", "clinicalAttributeId"]
)

if duplicates.any():
    raise RuntimeError(
        "Duplicate patient-method measurements detected. "
        "Resolve these before classification."
    )

print("\nMSI data validation: PASSED")

# ------------------------------------------------------------
# 4. Create patient-level score table
# ------------------------------------------------------------

scores = msi.pivot(
    index="patientId",
    columns="clinicalAttributeId",
    values="numeric_value"
)

scores = scores.rename(columns={
    "MSI_SCORE_MANTIS": "mantis_score",
    "MSI_SENSOR_SCORE": "msisensor_score"
})

# Track the original sample IDs used by each method.

sample_ids = msi.pivot(
    index="patientId",
    columns="clinicalAttributeId",
    values="sampleId"
)

sample_ids = sample_ids.rename(columns={
    "MSI_SCORE_MANTIS": "mantis_sample_id",
    "MSI_SENSOR_SCORE": "msisensor_sample_id"
})

patient_msi = scores.join(sample_ids)

# ------------------------------------------------------------
# 5. Restrict to patients with primary tumor RNA-seq
# ------------------------------------------------------------

primary_inventory = inventory[
    inventory["sample_type"] == "Primary Tumor"
].copy()

primary_patients = (
    primary_inventory[
        ["patient_id", "project"]
    ]
    .drop_duplicates()
)

# Ensure patients are not assigned to multiple projects.

project_conflicts = (
    primary_patients.groupby("patient_id")["project"]
    .nunique()
)

if (project_conflicts > 1).any():
    raise RuntimeError(
        "Patients assigned to multiple TCGA projects."
    )

primary_patients = primary_patients.rename(
    columns={
        "patient_id": "patientId",
        "project": "gdc_project"
    }
)

cohort = primary_patients.merge(
    patient_msi.reset_index(),
    on="patientId",
    how="left",
    validate="one_to_one"
)

print("\nPrimary tumor RNA-seq patients:", len(cohort))

# ------------------------------------------------------------
# 6. Evaluate measurement availability
# ------------------------------------------------------------

has_mantis = cohort["mantis_score"].notna()

has_sensor = cohort["msisensor_score"].notna()

has_both = has_mantis & has_sensor

same_sample = (
    has_both
    & (
        cohort["mantis_sample_id"]
        == cohort["msisensor_sample_id"]
    )
)

print("\nPatients with both scores:", has_both.sum())

print(
    "Both scores from matching cBioPortal samples:",
    same_sample.sum()
)

print(
    "Both scores but different sample IDs:",
    (has_both & ~same_sample).sum()
)

# ------------------------------------------------------------
# 7. Apply published MSI thresholds
# ------------------------------------------------------------

mantis_low = (
    cohort["mantis_score"] < MANTIS_THRESHOLD
)

sensor_low = (
    cohort["msisensor_score"] < MSISENSOR_THRESHOLD
)

mantis_high = (
    cohort["mantis_score"] >= MANTIS_THRESHOLD
)

sensor_high = (
    cohort["msisensor_score"] >= MSISENSOR_THRESHOLD
)

concordant_low = (
    same_sample
    & mantis_low
    & sensor_low
)

concordant_high = (
    same_sample
    & mantis_high
    & sensor_high
)

discordant = (
    same_sample
    & (
        (mantis_low & sensor_high)
        | (mantis_high & sensor_low)
    )
)

sample_mismatch = has_both & ~same_sample

incomplete = ~has_both

cohort["msi_group"] = np.select(
    [
        concordant_low,
        concordant_high,
        discordant,
        sample_mismatch,
        incomplete
    ],
    [
        "Concordant low MSI",
        "Concordant MSI-H",
        "Discordant",
        "Sample ID mismatch",
        "Incomplete"
    ],
    default="Unclassified"
)

# ------------------------------------------------------------
# 8. Display cohort distribution
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PRELIMINARY MSI CLASSIFICATION")
print("-" * 65)

classification_counts = (
    cohort["msi_group"]
    .value_counts(dropna=False)
)

print("\nOverall distribution:")

print(classification_counts.to_string())

# ------------------------------------------------------------
# 9. Project-specific distribution
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PROJECT-SPECIFIC MSI DISTRIBUTION")
print("-" * 65)

project_summary = pd.crosstab(
    cohort["gdc_project"],
    cohort["msi_group"]
)

print("\n")
print(project_summary.to_string())

# ------------------------------------------------------------
# 10. Preliminary research cohort
# ------------------------------------------------------------

preliminary_cohort = cohort[
    cohort["msi_group"] == "Concordant low MSI"
].copy()

print("\n" + "-" * 65)
print("PRELIMINARY LOW-MSI RESEARCH COHORT")
print("-" * 65)

print(
    "\nTotal candidate patients:",
    len(preliminary_cohort)
)

print("\nPatients by project:")

print(
    preliminary_cohort["gdc_project"]
    .value_counts()
    .to_string()
)

print("\nIMPORTANT:")
print(
    "These patients have concordant low MSI scores."
)

print(
    "They are not yet independently verified as MSS."
)

# ------------------------------------------------------------
# 11. Save results
# ------------------------------------------------------------

full_path = (
    METADATA_DIR /
    "tcga_msi_preliminary_patient_classification.csv"
)

cohort.to_csv(
    full_path,
    index=False
)

preliminary_path = (
    METADATA_DIR /
    "tcga_preliminary_low_msi_cohort.csv"
)

preliminary_cohort.to_csv(
    preliminary_path,
    index=False
)

summary_path = (
    RESULTS_DIR /
    "tcga_preliminary_msi_group_summary.csv"
)

project_summary.to_csv(summary_path)

# ------------------------------------------------------------
# 12. Record classification methodology
# ------------------------------------------------------------

methodology = {
    "project": "SurfaceTarget-CRC",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source": (
        "cBioPortal TCGA Colorectal "
        "PanCancer Atlas 2018"
    ),

    "classification_type": "Preliminary computational",

    "thresholds": {
        "MANTIS": MANTIS_THRESHOLD,
        "MSIsensor": MSISENSOR_THRESHOLD
    },

    "low_msi_rule": (
        "Both MANTIS < 0.4 and MSIsensor < 3.5, "
        "with matching cBioPortal sample IDs."
    ),

    "primary_tumor_rnaseq_required": True,

    "preliminary_low_msi_patients": int(
        len(preliminary_cohort)
    ),

    "limitations": [
        "Low MSI scores do not distinguish MSS from MSI-L.",
        "Computational classifications are not clinical validation.",
        "Independent MSI annotations remain to be evaluated.",
        "TCGA and cBioPortal cohort differences require documentation."
    ],

    "references": [
        "https://pmc.ncbi.nlm.nih.gov/articles/PMC5352334/",
        "https://pmc.ncbi.nlm.nih.gov/articles/PMC11317526/"
    ]
}

methodology_path = (
    METADATA_DIR /
    "tcga_preliminary_msi_methodology.json"
)

with open(methodology_path, "w") as f:
    json.dump(
        methodology,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 13. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 1.6 SUMMARY")
print("=" * 65)

print("\nFiles saved:")

print("1. tcga_msi_preliminary_patient_classification.csv")
print("2. tcga_preliminary_low_msi_cohort.csv")
print("3. tcga_preliminary_msi_group_summary.csv")
print("4. tcga_preliminary_msi_methodology.json")

print("\nNo expression matrices downloaded.")

print("\nPATCH 1.6 COMPLETE")